In [201]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [202]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [203]:
compact = False

In [204]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 1:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading 2024-08-31_00-36-28.parquet
Reading 2024-08-30_23-46-12.parquet
Reading 2024-08-31_00-40-38.parquet
Reading 2024-08-31_00-51-48.parquet
Reading 2024-08-31_00-27-50.parquet
Reading 2024-08-31_00-30-45.parquet
Reading 2024-08-31_00-41-46.parquet
Reading 2024-08-30_22-21-59.parquet
Reading 2024-08-31_00-32-34.parquet
Reading 2024-08-31_01-01-00.parquet
Reading 2024-08-31_00-25-13.parquet
Reading 2024-08-31_00-45-09.parquet
Reading 2024-08-31_00-24-14.parquet
Reading 2024-08-30_23-08-29.parquet
Reading 2024-08-30_23-34-09.parquet
Reading 2024-08-31_01-04-17.parquet
Reading 2024-08-31_00-39-37.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
4896996752,NaN,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.5130,0.015390,0,"[8, 4, 0, 0, 0]"
6158719728,4.896997e+09,"[41, 30, 42]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6077,0.024308,1,"[4, 8, 3, 2, 0]"
6158687296,6.158720e+09,"[41, 30, 42, 6]","[92, 92]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,8,0.5770,0.046160,1,"[4, 8, 6, 3, 0]"
6158493008,6.158687e+09,"[41, 30, 42, 6, 28]","[84, 84]",0,"[0, 0]","[16, 16]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.5421,0.086736,2,"[4, 2, 8, 0, 0]"
6158678432,6.158493e+09,"[41, 30, 42, 6, 28]","[84, 78]",0,"[0, 6]","[16, 22]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5421,0.102999,2,"[4, 2, 8, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6144387440,6.081101e+09,"[29, 10, 18, 6, 33]","[80, 104]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.9862,0.078896,4,"[7, 0, 0, 0, 0]"
6093877792,NaN,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.4805,0.014415,0,"[8, 2, 0, 0, 0]"
6096669040,6.093878e+09,"[32, 46, 43]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.3778,0.015112,0,"[8, 7, 6, 4, 2]"


In [205]:
raw_df.dtypes

prev_entry           float64
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

## Process data

In [206]:
df = Processor(raw_df).get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name
4896996752,e17a5c66-a631-4668-aa68-a9b3c23edae2,0,0,0,0,0,0,0,0,0,...,0,0,0,call,2,0,0.5130,0.015390,preflop,Tord
6158719728,e17a5c66-a631-4668-aa68-a9b3c23edae2,0,0,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.6077,0.024308,flop,Tord
6158687296,e17a5c66-a631-4668-aa68-a9b3c23edae2,0,4,0,0,0,2,0,0,0,...,0,0,0,raise,8,1,0.5770,0.046160,turn,Tord
6158493008,e17a5c66-a631-4668-aa68-a9b3c23edae2,0,4,8,0,0,2,0,0,0,...,0,0,0,check,0,1,0.5421,0.086736,river,Tord
6158678432,e17a5c66-a631-4668-aa68-a9b3c23edae2,0,4,8,0,0,2,0,0,0,...,0,0,0,call,6,1,0.5421,0.102999,river,Tord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6144387440,eb943ced-a7a7-46a1-a8e7-3d7f28fd9e93,0,0,4,0,0,2,0,0,0,...,0,0,0,check,0,4,0.9862,0.078896,river,Tord
6093877792,e4034a79-bf79-4c93-a08b-17e5ce1f61a3,0,0,0,0,0,0,0,0,0,...,0,0,0,call,2,0,0.4805,0.014415,preflop,Tord
6096669040,e4034a79-bf79-4c93-a08b-17e5ce1f61a3,0,0,0,0,0,2,0,0,0,...,0,0,0,check,0,0,0.3778,0.015112,flop,Tord
6442206016,e4034a79-bf79-4c93-a08b-17e5ce1f61a3,0,0,0,0,0,2,0,0,0,...,0,0,0,check,0,0,0.2393,0.009572,turn,Tord


In [207]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [208]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [209]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [210]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_call_showdown,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name
4896996752,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,call,2,preflop,Tord
6158719728,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,flop,Tord
6158687296,0,4,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,8,turn,Tord
6158493008,0,4,8,0,0,2,0,0,0,0,...,0,0,1,0,0,0,check,0,river,Tord
6158678432,0,4,8,0,0,2,0,0,0,0,...,0,0,1,0,0,0,call,6,river,Tord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6144387440,0,0,4,0,0,2,0,0,0,0,...,0,0,2,0,0,0,check,0,river,Tord
6093877792,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,call,2,preflop,Tord
6096669040,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,check,0,flop,Tord
6442206016,0,0,0,0,0,2,0,0,0,0,...,0,0,2,0,0,0,check,0,turn,Tord


In [211]:
y

4896996752    0
6158719728    1
6158687296    1
6158493008    1
6158678432    1
             ..
6144387440    4
6093877792    0
6096669040    0
6442206016    0
6093891136    0
Name: excess_rank, Length: 781, dtype: int64

In [212]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=1000
            ),
        ),
    ]
)

In [213]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (622, 34)
Test shape: (159, 34)


In [214]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.8364779874213837


In [215]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.8364779874213837
Mean goodness:  0.761623144726491


,0,1,2,3,4,5,true,pred,correct,goodness
0,0.958029,0.039328,7.999742e-04,0.000843,0.000262,0.000738,0,0,True,0.958029
1,0.210778,0.755067,2.310268e-02,0.003805,0.004009,0.003238,1,1,True,0.755067
2,0.993665,0.005305,1.565592e-04,0.000382,0.000287,0.000204,0,0,True,0.993665
3,0.865446,0.116945,3.453227e-03,0.005826,0.003091,0.005239,1,0,False,0.116945
4,0.242639,0.705275,2.247875e-02,0.003581,0.003447,0.022580,1,1,True,0.705275
...,...,...,...,...,...,...,...,...,...,...
154,0.162824,0.774175,1.422757e-02,0.005764,0.022661,0.020348,1,1,True,0.774175
155,0.179995,0.718740,3.244308e-03,0.005999,0.058883,0.033139,1,1,True,0.718740
156,0.936108,0.057366,1.586433e-03,0.001498,0.000766,0.002675,0,0,True,0.936108
157,0.997665,0.002317,2.468566e-07,0.000010,0.000006,0.000001,0,0,True,0.997665


In [216]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

26 incorrect predictions:


,0,1,2,3,4,5,true,pred,correct,goodness
3,0.865446,0.116945,0.003453,5.826475e-03,3.090676e-03,5.238982e-03,1,0,False,1.169449e-01
5,0.873802,0.114928,0.008785,6.263632e-04,4.574439e-04,1.400401e-03,1,0,False,1.149283e-01
6,0.985071,0.014894,0.000014,1.514532e-05,4.761365e-06,1.163565e-06,1,0,False,1.489369e-02
8,0.826244,0.099876,0.073880,7.279056e-07,4.928545e-13,2.075373e-10,1,0,False,9.987561e-02
10,0.041431,0.958559,0.000010,6.826352e-11,1.788032e-13,3.268010e-10,3,1,False,6.826352e-11
22,0.190667,0.688208,0.025069,2.065945e-02,9.872338e-03,6.552438e-02,4,1,False,9.872338e-03
56,0.461935,0.475785,0.006074,4.530667e-03,7.190911e-03,4.448364e-02,0,1,False,4.619354e-01
57,0.160662,0.656825,0.001839,8.700986e-05,5.415360e-02,1.264342e-01,5,1,False,1.264342e-01
66,0.500237,0.483594,0.011492,1.274098e-03,2.875774e-03,5.268801e-04,1,0,False,4.835937e-01
68,0.164730,0.816931,0.000343,8.301347e-04,1.614175e-02,1.025211e-03,0,1,False,1.647295e-01
